##### Copyright 2025 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 音訊理解

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/gemma/docs/capabilities/audio"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
</td> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/docs/capabilities/audio.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemma/cookbook/blob/main/docs/capabilities/audio.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemma%2Fcookbook%2Fmain%2Fdocs%2Fcapabilities%2Faudio.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemma/cookbook/blob/main/docs/capabilities/audio.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td>
</table>

從 [Gemma 3n](https://ai.google.dev/gemma/docs/gemma-3n) 開始，您可以直接在 prompt 和工作流程中使用音訊。音訊和口語是豐富的資料來源，可用於捕捉使用者意圖、記錄有關我們周圍世界的資訊以及理解
需要解决的具体问题。
本指南概述了 [Gemma 4](https://ai.google.dev/gemma/docs/core) 的音訊處理功能，包括自動語音辨識 (ASR)、翻譯和一般語音理解。

這個notebook將在 T4 GPU 上執行。

## 安裝 Python 軟體包

安裝執行 Gemma 模型和發出請求所需的 Hugging Face 庫。

In [ ]:
# Install PyTorch & other libraries
!pip install torch accelerate

# Install the transformers library
!pip install "transformers>=5.5.0"

## 負載模型

使用`transformers` 庫透過`AutoProcessor` 和`AutoModelForImageTextToText` 類別建立`processor` 和`model` 的實例，如下列程式碼範例所示：

In [2]:
MODEL_ID = "google/gemma-4-E2B-it" # @param ["google/gemma-4-E2B-it","google/gemma-4-E4B-it", "google/gemma-4-12B-it"]

from transformers import pipeline

pipe = pipeline(
    task="any-to-any",
    model=MODEL_ID,
    device_map="auto",
    dtype="auto"
)

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

## 音訊數據

數位音訊資料可以有多種格式和解析度等級。 Gemma 可使用的實際音訊格式（例如 MP3 和 WAV 格式）由您選擇將聲音資料轉換為張量的 framework 決定。以下是準備使用 Gemma 處理的音訊資料的一些具體注意事項：
-   **token 成本：** 對於 Gemma 4，每秒音訊為 25 tokens。 （對於 Gemma 3n，每秒音訊為 6.25 tokens）。
-   **剪輯長度：** 音訊支援的最大長度為 30 秒。
-   **音訊通道：** 音訊資料以單一音訊通道進行處理。
    If you are using multi-channel audio, such as left and right channels,
    consider reducing the data to a single channel by removing channels or
    combining the sound data into a single channel.
-   **技術編碼：**
- **取樣率：** 16kHz - **位元深度：** 32 位元浮點格式，樣本在 [-1, 1] 範圍內標準化。
如果您計劃處理的音訊資料與輸入明顯不同
處理，特別是在通道、取樣率和位元深度方面，
考慮重新採樣或修剪音訊資料以匹配數據
模型處理的解析度。

## 音訊編碼

雖然高級庫（例如 Hugging Face `AutoProcessor`）通常會自動處理音訊預處理，但有時您可能需要實作自訂編碼。
當使用您自己的程式碼實作對音訊資料進行編碼以便與 Gemma 一起使用時，您應該遵循建議的轉換過程。如果你正在工作
以特定格式編碼的音訊文件，例如 MP3 或 WAV 編碼數據，
您必須先使用library（例如`ffmpeg`）將它們解碼為樣本。曾經
資料被解碼，將音訊轉換為單聲道、16 kHz float32
波形在 [-1, 1] 範圍內。例如，如果您正在使用立體聲
若要以 44.1 kHz 簽署 16 位元 PCM 整數 WAV 文件，請依照下列步驟操作：
*   將音訊資料重新取樣至 16 kHz
*   透過平均 2 個聲道將立體聲縮混為單聲道
*   從 int16 轉換為 float32，並除以 32768.0 以縮放到範圍
    [-1, 1]

> 注意：將音訊重新取樣至 16 kHz 時，應使用傅立葉方法以獲得最佳結果，例如 `scipy.signal.resample` 或 `librosa.sample(res_type ='scipy')`。

## 語音轉文字

Gemma 4 E2B、E4B 和 12B Unified 經過多語言語音辨識訓練，讓您將各種語言的音訊輸入轉錄為文字。
使用以下 prompt 結構進行**音訊語音辨識 (ASR)**。
```text
Transcribe the following speech segment in {LANGUAGE} into {LANGUAGE} text.

Follow these specific instructions for formatting the answer:
*   Only output the transcription, with no newlines.
*   When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 3 instead of three.
```

以下程式碼範例示範如何prompt模型以使用Hugging Face Transformers從音訊檔案轉錄文字：

In [6]:
from transformers import GenerationConfig
config = GenerationConfig.from_pretrained(MODEL_ID)
config.max_new_tokens = 64
gen_kwargs = dict(generation_config=config)

RESOURCE_URL_PREFIX = "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/apps/sample-data/"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Transcribe the following speech segment in its original language. Follow these specific instructions for formatting the answer:\n* Only output the transcription, with no newlines.\n* When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 3 instead of three."},
            #{"type": "text", "text": "Transcribe the following speech segment in English into English text. Follow these specific instructions for formatting the answer:\n* Only output the transcription, with no newlines.\n* When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 3 instead of three."},
            {"type": "audio", "audio": f"{RESOURCE_URL_PREFIX}journal1.wav"},
        ]
    }
]

outputs = pipe(messages, return_full_text=False, generate_kwargs=gen_kwargs)
print(outputs[0]['generated_text'])

I woke up early today feeling really fresh the morning light was beautiful and I enjoyed a nice cup of coffee<turn|>


In [7]:
from transformers import GenerationConfig
config = GenerationConfig.from_pretrained(MODEL_ID)
config.max_new_tokens = 1024
gen_kwargs = dict(generation_config=config)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Give me a concise overview of these audio files."},
            {"type": "text", "text": "journal1:"},
            {"type": "audio", "audio": f"{RESOURCE_URL_PREFIX}journal1.wav"},
            {"type": "text", "text": "journal2:"},
            {"type": "audio", "audio": f"{RESOURCE_URL_PREFIX}journal2.wav"},
            {"type": "text", "text": "journal3:"},
            {"type": "audio", "audio": f"{RESOURCE_URL_PREFIX}journal3.wav"},
            {"type": "text", "text": "journal4:"},
            {"type": "audio", "audio": f"{RESOURCE_URL_PREFIX}journal4.wav"},
            {"type": "text", "text": "journal5:"},
            {"type": "audio", "audio": f"{RESOURCE_URL_PREFIX}journal5.wav"},
        ]
    }
]

outputs = pipe(messages, return_full_text=False, generate_kwargs=gen_kwargs)
print(outputs[0]['generated_text'])

Here is a concise overview of each audio file:

**journal1:** The speaker describes a fresh and peaceful day, enjoying a cup of coffee.
**journal2:** The speaker had a perfect day at the park, including a walk and watching cherry blossoms.
**journal3:** The speaker finished the day with a good book, feeling grateful for simple moments.
**journal4:** The speaker returned from work and noted the beautiful night sky and a clear view from the train.
**journal5:** The speaker had a great lunch with an old friend, which was a pleasant way to catch up and made their day.
<turn|>


## 自動語音翻譯

Gemma 4 E2B、E4B 和 12B Unified 經過多語言語音翻譯任務訓練，讓您可以將口語音訊直接翻譯成另一種語言。
使用以下 prompt 結構進行**自動語音翻譯 (AST)**。
```text
Transcribe the following speech segment in {SOURCE_LANGUAGE}, then translate it into {TARGET_LANGUAGE}.
When formatting the answer, first output the transcription in {SOURCE_LANGUAGE}, then one newline, then output the string '{TARGET_LANGUAGE}: ', then the translation in {TARGET_LANGUAGE}.
```

以下程式碼範例展示如何prompt模型以使用Hugging Face Transformers將語音音訊轉換為文字：

In [8]:
from transformers import GenerationConfig
config = GenerationConfig.from_pretrained(MODEL_ID)
config.max_new_tokens = 64
gen_kwargs = dict(generation_config=config)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Transcribe the following speech segment in English, then translate it into Korean. When formatting the answer, first output the transcription in English, then one newline, then output the string 'Korean: ', then the translation in Korean."},
            {"type": "audio", "audio": "https://ai.google.dev/gemma/docs/audio/roses-are.wav"},
        ]
    }
]

outputs = pipe(messages, return_full_text=False, generate_kwargs=gen_kwargs)
print(outputs[0]['generated_text'])

Roses are red, violets are blue.
Korean: 장미는 빨갛고, 제비꽃은 파랗다.<turn|>


## 自動語音翻譯/自動語音識別

自己嘗試這個

In [9]:
!pip install ipywebrtc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.7/260.7 kB 21.8 MB/s eta 0:00:00


按下圓圈按鈕並開始講話。完成後再次點選圓圈按鈕。小部件將立即開始播放它捕獲的內容。

In [11]:
from google.colab import output
output.enable_custom_widget_manager()

from ipywebrtc import AudioRecorder, CameraStream

camera = CameraStream(constraints={'audio': True,'video':False})
recorder = AudioRecorder(stream=camera)
recorder

AudioRecorder(audio=Audio(value=b'', format='webm'), stream=CameraStream(constraints={'audio': True, 'video': …

將 webm 檔案轉換為PyTorch可以理解的 wav 格式。

In [12]:
with open('/content/recording.webm', 'wb') as f:
    f.write(recorder.audio.value)
!ffmpeg -i /content/recording.webm /content/recording.wav -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

### 自動語音識別

In [13]:
from transformers import GenerationConfig
config = GenerationConfig.from_pretrained(MODEL_ID)
config.max_new_tokens = 64
gen_kwargs = dict(generation_config=config)

messages = [{
  "role": "user",
  "content": [
    {"type": "text", "text": "Transcribe the following speech segment in its original language. Follow these specific instructions for formatting the answer:\n* Only output the transcription, with no newlines.\n* When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 3 instead of three."},
    {"type": "audio", "audio": "/content/recording.wav"},
  ]
}]

outputs = pipe(messages, return_full_text=False, generate_kwargs=gen_kwargs)
print(outputs[0]['generated_text'])

How can I get to the station?<turn|>


### 穀草轉氨酶

In [14]:
messages = [{
  "role": "user",
  "content": [
    {"type": "text", "text": "Transcribe the following speech segment in English, then translate it into Korean. When formatting the answer, first output the transcription in English, then one newline, then output the string 'Korean: ', then the translation in Korean."},
    {"type": "audio", "audio": "/content/recording.wav"},
  ]
}]

outputs = pipe(messages, return_full_text=False, generate_kwargs=gen_kwargs)
print(outputs[0]['generated_text'])

How can I get to the station?
Korean: 역에 어떻게 가나요?<turn|>


## 摘要與後續步驟

在本指南中，您學習如何使用 Gemma 4 模型處理音訊。這些範例示範如何執行語音轉文字 (ASR) 來轉錄口語，以及如何執行自動語音翻譯 (AST) 將口語音訊直接翻譯成另一種語言。您還了解如何在 notebook 環境中從麥克風捕獲音訊進行處理。
查看以下文件以進一步閱讀。
- [執行Gemma概述](https://ai.google.dev/gemma/docs/run)
- [視覺理解](https://ai.google.dev/gemma/docs/capabilities/vision)
- [思維模式](https://ai.google.dev/gemma/docs/capabilities/thinking)
